In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install transformers==4.51.3
    !pip install --no-deps unsloth

In [ ]:
import torch
from unsloth import FastLanguageModel

repo_id="barryallen16/Qwen2.5_7B_java_vul_v3"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=repo_id,
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
def classify_code(code):
    messages = [
        {
            "role": "system",
            "content": "You are a code vulnerability classifier. Analyze the provided Java code and classify it as either 'VULNERABLE' or 'NOT VULNERABLE'. If vulnerable, specify the exact vulnerability type from the dataset. Respond strictly in this format: 'Classification: <VULNERABLE/NOT VULNERABLE>. Vulnerability Type: <type>'. Do not include reasoning.",
        },
        {"role": "user", "content": code},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=256
    )

    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    last_user_msg = messages[-1]["content"]
    response = output_text.split(last_user_msg)[-1].strip()

    return response


In [ ]:
import json
def evaluate_model(llm_name,test_cases):
    results = []
    correct = 0

    for i, test_case in enumerate(test_cases):
        code = test_case["code"]
        expected = test_case["expected"]
        # print(f"\nEvaluating test case {i+1}...")
        # print("Code snippet:")
        # print(code[:200] + "..." if len(code) > 200 else code)
        try:
            prediction = classify_code(code)
            # print("Model output:", prediction)
            if "Classification: VULNERABLE" in prediction:
                predicted_class = "VULNERABLE"
            elif "Classification: NOT_VULNERABLE" in prediction:
                predicted_class = "NOT_VULNERABLE"
            else:
                predicted_class = "UNKNOWN"
            is_correct = predicted_class == expected
            if is_correct:
                correct += 1

            results.append({
                "test_case": i+1,
                "expected": expected,
                "predicted": predicted_class,
                "correct": is_correct,
                "model_output": prediction
            })

        except Exception as e:
            print(f"Error processing test case {i+1}: {str(e)}")
            results.append({
                "test_case": i+1,
                "error": str(e)
            })

    accuracy = correct / len(test_cases) * 100
    print(f"\n{llm_name} Testcases Evaluation Summary:")
    print(f"Total test cases: {len(test_cases)}")
    print(f"Correct predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
    output_file_path=llm_name+"_evaluation_results.json"
    with open(output_file_path, "w") as f:
        json.dump({
            "summary": {
                "total_test_cases": len(test_cases),
                "correct_predictions": correct,
                "accuracy": accuracy
            },
            "detailed_results": results
        }, f, indent=2)

    return results


In [ ]:
# @title
deepseek_test=[
    {
        "code": 'public class Example1 {\n    public static void main(String[] args) {\n        String username = "admin";\n        String password = "admin123";\n        if (args[0].equals(username) && args[1].equals(password)) {\n            System.out.println("Access granted");\n        }\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example2 {\n    public static void main(String[] args) {\n        String query = "SELECT * FROM users WHERE username = \'" + args[0] + "\'";\n        Statement stmt = connection.createStatement();\n        ResultSet rs = stmt.executeQuery(query);\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example3 {\n    public static void main(String[] args) {\n        String input = args[0];\n        System.out.println("<div>" + input + "</div>");\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example4 {\n    public static void main(String[] args) throws Exception {\n        String fileName = "../../etc/passwd";\n        File file = new File("uploads/" + fileName);\n        FileInputStream fis = new FileInputStream(file);\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example5 {\n    public static void main(String[] args) {\n        String secretKey = "A1B2C3D4E5F6";\n        Cipher cipher = Cipher.getInstance("AES");\n        cipher.init(Cipher.ENCRYPT_MODE, new SecretKeySpec(secretKey.getBytes(), "AES"));\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example6 {\n    public static void main(String[] args) {\n        String username = System.getenv("DB_USER");\n        String password = System.getenv("DB_PASS");\n        String query = "SELECT * FROM users WHERE username = ?";\n        PreparedStatement stmt = connection.prepareStatement(query);\n        stmt.setString(1, args[0]);\n        ResultSet rs = stmt.executeQuery();\n    }\n}',
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": 'public class Example7 {\n    public static void main(String[] args) {\n        String input = args[0];\n        String safeInput = input.replaceAll("<", "&lt;").replaceAll(">", "&gt;");\n        System.out.println("<div>" + safeInput + "</div>");\n    }\n}',
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": 'public class Example8 {\n    public static void main(String[] args) throws Exception {\n        String fileName = args[0];\n        Path safePath = Paths.get("uploads/").resolve(Paths.get(fileName)).normalize();\n        if (!safePath.startsWith("uploads/")) {\n            throw new IllegalArgumentException("Invalid file path");\n        }\n        FileInputStream fis = new FileInputStream(safePath.toFile());\n    }\n}',
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": 'public class Example9 {\n    public static void main(String[] args) {\n        String algorithm = "AES/CBC/PKCS5Padding";\n        KeyGenerator keyGen = KeyGenerator.getInstance("AES");\n        keyGen.init(256);\n        SecretKey secretKey = keyGen.generateKey();\n        Cipher cipher = Cipher.getInstance(algorithm);\n        cipher.init(Cipher.ENCRYPT_MODE, secretKey);\n    }\n}',
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": 'public class Example10 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        if (userInput.matches("[a-zA-Z0-9]+")) {\n            String query = "SELECT * FROM users WHERE username = \'" + userInput + "\'";\n            Statement stmt = connection.createStatement();\n            ResultSet rs = stmt.executeQuery(query);\n        }\n    }\n}',
        "expected": "VULNERABLE"
    }
]
result=evaluate_model("deepseek",deepseek_test)


In [ ]:
# @title
gemini_testcases=[
    {
        "code": "import java.sql.Connection;\nimport java.sql.DriverManager;\nimport java.sql.Statement;\n\npublic class SQLInjectionVulnerable {\n    public void queryDatabase(String userInput) throws Exception {\n        Connection con = DriverManager.getConnection(\"jdbc:mysql://localhost:3306/db\", \"user\", \"password\");\n        Statement stmt = con.createStatement();\n        String query = \"SELECT * FROM users WHERE username = '\" + userInput + \"'\";\n        stmt.executeQuery(query);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.sql.Connection;\nimport java.sql.DriverManager;\nimport java.sql.PreparedStatement;\n\npublic class SQLInjectionNotVulnerable {\n    public void queryDatabase(String userInput) throws Exception {\n        Connection con = DriverManager.getConnection(\"jdbc:mysql://localhost:3306/db\", \"user\", \"password\");\n        String query = \"SELECT * FROM users WHERE username = ?\";\n        PreparedStatement pstmt = con.prepareStatement(query);\n        pstmt.setString(1, userInput);\n        pstmt.executeQuery();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.io.File;\n\npublic class PathTraversalVulnerable {\n    public void readFile(String fileName) {\n        File file = new File(\"/var/www/html/\" + fileName);\n        // ... code to read file\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.File;\n\npublic class PathTraversalNotVulnerable {\n    public void readFile(String fileName) {\n        File file = new File(fileName);\n        if (!file.getAbsolutePath().startsWith(\"/var/www/html/\")) {\n            throw new SecurityException(\"Access Denied\");\n        }\n        // ... code to read file\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import javax.servlet.http.HttpServletRequest;\nimport javax.servlet.http.HttpServletResponse;\nimport java.io.IOException;\n\npublic class XSSVulnerable {\n    protected void doGet(HttpServletRequest request, HttpServletResponse response) throws IOException {\n        String comment = request.getParameter(\"comment\");\n        response.getWriter().println(\"<html><body>\" + comment + \"</body></html>\");\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import javax.servlet.http.HttpServletRequest;\nimport javax.servlet.http.HttpServletResponse;\nimport java.io.IOException;\nimport org.owasp.encoder.Encode;\n\npublic class XSSNotVulnerable {\n    protected void doGet(HttpServletRequest request, HttpServletResponse response) throws IOException {\n        String comment = request.getParameter(\"comment\");\n        response.getWriter().println(\"<html><body>\" + Encode.forHtml(comment) + \"</body></html>\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class HardcodedPasswordVulnerable {\n    public void connectToDb() {\n        String dbPassword = \"mySuperSecretPassword123\";\n        // ... code to connect to database\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.util.Properties;\nimport java.io.FileInputStream;\nimport java.io.IOException;\n\npublic class HardcodedPasswordNotVulnerable {\n    public String getDbPassword() throws IOException {\n        Properties props = new Properties();\n        FileInputStream fis = new FileInputStream(\"config.properties\");\n        props.load(fis);\n        return props.getProperty(\"db.password\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import javax.xml.parsers.DocumentBuilder;\nimport javax.xml.parsers.DocumentBuilderFactory;\nimport org.xml.sax.InputSource;\nimport java.io.StringReader;\n\npublic class XXEVulnerable {\n    public void parseXML(String xml) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        builder.parse(new InputSource(new StringReader(xml)));\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import javax.xml.parsers.DocumentBuilder;\nimport javax.xml.parsers.DocumentBuilderFactory;\nimport org.xml.sax.InputSource;\nimport java.io.StringReader;\n\npublic class XXENotVulnerable {\n    public void parseXML(String xml) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        factory.setFeature(\"http://apache.org/xml/features/disallow-doctype-decl\", true);\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        builder.parse(new InputSource(new StringReader(xml)));\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.io.*;\n\npublic class DeserializationVulnerable {\n    public void process(InputStream input) throws IOException, ClassNotFoundException {\n        ObjectInputStream ois = new ObjectInputStream(input);\n        Object obj = ois.readObject();\n        // ...\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\n\npublic class DeserializationNotVulnerable {\n    // Using a safe serialization format like JSON with a library like Jackson\n    // is preferred over native Java serialization.\n    public void process(String jsonInput) {\n        // Use a library like Jackson or Gson to parse JSON safely.\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.util.Random;\n\npublic class InsecureRandomVulnerable {\n    public int generateToken() {\n        Random rand = new Random();\n        return rand.nextInt();\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.security.SecureRandom;\n\npublic class InsecureRandomNotVulnerable {\n    public int generateToken() {\n        SecureRandom rand = new SecureRandom();\n        return rand.nextInt();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class CommandInjectionVulnerable {\n    public void runCommand(String userInput) throws java.io.IOException {\n        Runtime.getRuntime().exec(\"cmd /c dir \" + userInput);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class CommandInjectionNotVulnerable {\n    public void runCommand(String userInput) throws java.io.IOException {\n        String[] command = {\"cmd\", \"/c\", \"dir\", userInput};\n        Runtime.getRuntime().exec(command);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    }
]

result=evaluate_model("gemini",gemini_testcases)

In [ ]:
# @title
claude_testcases=[
    {
        "code": "public class SQLInjectionVulnerable {\n    public void getUserData(String userId) {\n        String query = \"SELECT * FROM users WHERE id = '\" + userId + \"'\";\n        // Execute query without preparation\n        executeQuery(query);\n    }\n    private void executeQuery(String query) { /* implementation */ }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SQLInjectionSecure {\n    public void getUserData(String userId) {\n        String query = \"SELECT * FROM users WHERE id = ?\";\n        PreparedStatement stmt = connection.prepareStatement(query);\n        stmt.setString(1, userId);\n        stmt.executeQuery();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class PathTraversalVulnerable {\n    public void readFile(String fileName) throws IOException {\n        File file = new File(\"/uploads/\" + fileName);\n        FileInputStream fis = new FileInputStream(file);\n        // Process file\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class PathTraversalSecure {\n    public void readFile(String fileName) throws IOException {\n        if (fileName.contains(\"..\") || fileName.contains(\"/\") || fileName.contains(\"\\\\\")) {\n            throw new SecurityException(\"Invalid file name\");\n        }\n        File file = new File(\"/uploads/\" + fileName);\n        FileInputStream fis = new FileInputStream(file);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class XSSVulnerable {\n    public void displayUserInput(String userInput) {\n        String html = \"<div>\" + userInput + \"</div>\";\n        response.getWriter().println(html);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class XSSSecure {\n    public void displayUserInput(String userInput) {\n        String sanitized = StringEscapeUtils.escapeHtml4(userInput);\n        String html = \"<div>\" + sanitized + \"</div>\";\n        response.getWriter().println(html);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class HardcodedCredentials {\n    private static final String DB_PASSWORD = \"admin123\";\n    private static final String API_KEY = \"sk-1234567890abcdef\";\n    \n    public void connect() {\n        Connection conn = DriverManager.getConnection(\"jdbc:mysql://localhost/db\", \"admin\", DB_PASSWORD);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureCredentials {\n    private String dbPassword;\n    private String apiKey;\n    \n    public SecureCredentials() {\n        this.dbPassword = System.getenv(\"DB_PASSWORD\");\n        this.apiKey = System.getenv(\"API_KEY\");\n    }\n    \n    public void connect() {\n        Connection conn = DriverManager.getConnection(\"jdbc:mysql://localhost/db\", \"admin\", dbPassword);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class WeakCrypto {\n    public String encryptData(String data) throws Exception {\n        Cipher cipher = Cipher.getInstance(\"DES\");\n        SecretKeySpec key = new SecretKeySpec(\"12345678\".getBytes(), \"DES\");\n        cipher.init(Cipher.ENCRYPT_MODE, key);\n        return Base64.getEncoder().encodeToString(cipher.doFinal(data.getBytes()));\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class StrongCrypto {\n    public String encryptData(String data) throws Exception {\n        Cipher cipher = Cipher.getInstance(\"AES/GCM/NoPadding\");\n        SecretKeySpec key = new SecretKeySpec(generateSecureKey(), \"AES\");\n        cipher.init(Cipher.ENCRYPT_MODE, key);\n        return Base64.getEncoder().encodeToString(cipher.doFinal(data.getBytes()));\n    }\n    \n    private byte[] generateSecureKey() {\n        SecureRandom random = new SecureRandom();\n        byte[] key = new byte[32];\n        random.nextBytes(key);\n        return key;\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class InsecureRandom {\n    public String generateToken() {\n        Random random = new Random();\n        return String.valueOf(random.nextLong());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureRandom {\n    public String generateToken() {\n        java.security.SecureRandom random = new java.security.SecureRandom();\n        byte[] bytes = new byte[32];\n        random.nextBytes(bytes);\n        return Base64.getEncoder().encodeToString(bytes);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class CommandInjectionVulnerable {\n    public void executeCommand(String userInput) throws IOException {\n        String command = \"ls -la \" + userInput;\n        Runtime.getRuntime().exec(command);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class CommandInjectionSecure {\n    public void executeCommand(String userInput) throws IOException {\n        if (!userInput.matches(\"^[a-zA-Z0-9_-]+$\")) {\n            throw new IllegalArgumentException(\"Invalid input\");\n        }\n        String[] command = {\"ls\", \"-la\", userInput};\n        Runtime.getRuntime().exec(command);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class LDAPInjectionVulnerable {\n    public void searchUser(String username) throws NamingException {\n        String filter = \"(uid=\" + username + \")\";\n        DirContext ctx = new InitialDirContext();\n        ctx.search(\"ou=users,dc=example,dc=com\", filter, new SearchControls());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class LDAPInjectionSecure {\n    public void searchUser(String username) throws NamingException {\n        String escapedUsername = username.replace(\"\\\\\", \"\\\\\\\\\").replace(\"*\", \"\\\\*\").replace(\"(\", \"\\\\(\").replace(\")\", \"\\\\)\");\n        String filter = \"(uid=\" + escapedUsername + \")\";\n        DirContext ctx = new InitialDirContext();\n        ctx.search(\"ou=users,dc=example,dc=com\", filter, new SearchControls());\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class TrustAllCertificates {\n    public void makeHttpsRequest() throws Exception {\n        TrustManager[] trustAllCerts = new TrustManager[] {\n            new X509TrustManager() {\n                public X509Certificate[] getAcceptedIssuers() { return null; }\n                public void checkClientTrusted(X509Certificate[] certs, String authType) { }\n                public void checkServerTrusted(X509Certificate[] certs, String authType) { }\n            }\n        };\n        SSLContext sc = SSLContext.getInstance(\"SSL\");\n        sc.init(null, trustAllCerts, new java.security.SecureRandom());\n        HttpsURLConnection.setDefaultSSLSocketFactory(sc.getSocketFactory());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class ProperSSLValidation {\n    public void makeHttpsRequest() throws Exception {\n        SSLContext sslContext = SSLContext.getInstance(\"TLS\");\n        sslContext.init(null, null, null); // Use default trust managers\n        HttpsURLConnection connection = (HttpsURLConnection) url.openConnection();\n        connection.setSSLSocketFactory(sslContext.getSocketFactory());\n        connection.connect();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class XMLExternalEntity {\n    public void parseXML(String xmlContent) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        Document doc = builder.parse(new ByteArrayInputStream(xmlContent.getBytes()));\n        // Process document\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureXMLParsing {\n    public void parseXML(String xmlContent) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        factory.setFeature(\"http://apache.org/xml/features/disallow-doctype-decl\", true);\n        factory.setFeature(\"http://xml.org/sax/features/external-general-entities\", false);\n        factory.setFeature(\"http://xml.org/sax/features/external-parameter-entities\", false);\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        Document doc = builder.parse(new ByteArrayInputStream(xmlContent.getBytes()));\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class InsecureDeserialization {\n    public Object deserializeObject(byte[] data) throws Exception {\n        ObjectInputStream ois = new ObjectInputStream(new ByteArrayInputStream(data));\n        return ois.readObject();\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureDeserialization {\n    public Object deserializeObject(byte[] data) throws Exception {\n        ObjectInputStream ois = new ObjectInputStream(new ByteArrayInputStream(data)) {\n            @Override\n            protected Class<?> resolveClass(ObjectStreamClass desc) throws IOException, ClassNotFoundException {\n                if (!desc.getName().equals(\"com.example.SafeClass\")) {\n                    throw new InvalidClassException(\"Unauthorized deserialization attempt\", desc.getName());\n                }\n                return super.resolveClass(desc);\n            }\n        };\n        return ois.readObject();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class WeakHashFunction {\n    public String hashPassword(String password) throws NoSuchAlgorithmException {\n        MessageDigest md = MessageDigest.getInstance(\"MD5\");\n        byte[] hash = md.digest(password.getBytes());\n        return Base64.getEncoder().encodeToString(hash);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class StrongHashFunction {\n    public String hashPassword(String password) throws NoSuchAlgorithmException {\n        MessageDigest md = MessageDigest.getInstance(\"SHA-256\");\n        SecureRandom random = new SecureRandom();\n        byte[] salt = new byte[16];\n        random.nextBytes(salt);\n        md.update(salt);\n        byte[] hash = md.digest(password.getBytes());\n        return Base64.getEncoder().encodeToString(hash);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class LogInjection {\n    public void logUserInput(String userInput) {\n        logger.info(\"User input: \" + userInput);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureLogging {\n    public void logUserInput(String userInput) {\n        String sanitized = userInput.replaceAll(\"[\\\\r\\\\n]\", \"_\");\n        logger.info(\"User input: {}\", sanitized);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class FileUploadVulnerable {\n    public void uploadFile(String fileName, byte[] content) throws IOException {\n        File file = new File(\"/uploads/\" + fileName);\n        FileOutputStream fos = new FileOutputStream(file);\n        fos.write(content);\n        fos.close();\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class FileUploadSecure {\n    public void uploadFile(String fileName, byte[] content) throws IOException {\n        if (!fileName.matches(\"^[a-zA-Z0-9._-]+\\\\.(jpg|jpeg|png|gif|pdf|txt)$\")) {\n            throw new IllegalArgumentException(\"Invalid file type\");\n        }\n        if (content.length > 1024 * 1024) { // 1MB limit\n            throw new IllegalArgumentException(\"File too large\");\n        }\n        File file = new File(\"/uploads/\" + UUID.randomUUID() + \"_\" + fileName);\n        FileOutputStream fos = new FileOutputStream(file);\n        fos.write(content);\n        fos.close();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class ReflectionVulnerable {\n    public void executeMethod(String className, String methodName) throws Exception {\n        Class<?> clazz = Class.forName(className);\n        Method method = clazz.getMethod(methodName);\n        method.invoke(clazz.newInstance());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class ReflectionSecure {\n    private static final Set<String> ALLOWED_CLASSES = Set.of(\"com.example.SafeClass\");\n    \n    public void executeMethod(String className, String methodName) throws Exception {\n        if (!ALLOWED_CLASSES.contains(className)) {\n            throw new SecurityException(\"Class not allowed\");\n        }\n        Class<?> clazz = Class.forName(className);\n        Method method = clazz.getMethod(methodName);\n        method.invoke(clazz.newInstance());\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class BasicMathOperation {\n    public int add(int a, int b) {\n        return a + b;\n    }\n    \n    public int multiply(int a, int b) {\n        return a * b;\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class StringUtility {\n    public String reverse(String input) {\n        if (input == null) {\n            return null;\n        }\n        return new StringBuilder(input).reverse().toString();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    }
]
result=evaluate_model("claude",claude_testcases)

In [ ]:
# @title
qwen_testcases=[
    {
        "code": "public class Example1 {\n    public static void main(String[] args) {\n        String secretKey = \"ABCD1234\";\n        System.out.println(\"Secret: \" + secretKey);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example2 {\n    public static void main(String[] args) {\n        try {\n            Runtime.getRuntime().exec(\"rm -rf /tmp/*\");\n        } catch (Exception e) {\n            e.printStackTrace();\n        }\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example3 {\n    public static void main(String[] args) throws IOException {\n        FileInputStream fis = new FileInputStream(\"/tmp/data.txt\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example4 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        System.out.println(\"Input: \" + userInput);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example5 {\n    public static void main(String[] args) throws Exception {\n        ObjectInputStream ois = new ObjectInputStream(new FileInputStream(args[0]));\n        Object obj = ois.readObject();\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example6 {\n    public static void main(String[] args) {\n        final String apiKey = System.getenv(\"API_KEY\");\n        System.out.println(\"Using API key...\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example7 {\n    public static void main(String[] args) {\n        if (args.length > 0 && args[0].equals(\"admin\")) {\n            System.out.println(\"Access granted\");\n        }\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example8 {\n    public static void main(String[] args) {\n        String password = \"password123\";\n        System.out.println(\"Password: \" + password);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example9 {\n    public static void main(String[] args) throws Exception {\n        String filename = args[0];\n        if (!filename.endsWith(\".txt\")) {\n            throw new IllegalArgumentException(\"Only .txt files allowed\");\n        }\n        BufferedReader br = new BufferedReader(new FileReader(filename));\n        String line;\n        while ((line = br.readLine()) != null) {\n            System.out.println(line);\n        }\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example10 {\n    public static void main(String[] args) {\n        ProcessBuilder pb = new ProcessBuilder(args);\n        try {\n            pb.start();\n        } catch (Exception e) {\n            e.printStackTrace();\n        }\n    }\n}",
        "expected": "VULNERABLE"
    }
]
result=evaluate_model("qwen", qwen_testcases)

In [ ]:
# @title
grok_testcases=[
    {
        "code": "public class Example1 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        Runtime.getRuntime().exec(userInput);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example2 {\n    public static void main(String[] args) {\n        String safeString = \"Hello, World!\";\n        System.out.println(safeString);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example3 {\n    public static void main(String[] args) {\n        String query = \"SELECT * FROM users WHERE id = '\" + args[0] + \"'\";\n        Statement stmt = connection.createStatement();\n        stmt.executeQuery(query);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example4 {\n    public static void main(String[] args) {\n        String input = args[0];\n        if (input.matches(\"[a-zA-Z0-9]+\")) {\n            System.out.println(input);\n        } else {\n            throw new IllegalArgumentException(\"Invalid input\");\n        }\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example5 {\n    public static void main(String[] args) throws Exception {\n        String path = \"../../etc/passwd\";\n        File file = new File(path);\n        FileInputStream fis = new FileInputStream(file);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example6 {\n    public static void main(String[] args) {\n        int number = Integer.parseInt(args[0]);\n        System.out.println(\"Number: \" + number);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example7 {\n    public static void main(String[] args) {\n        String[] allowedValues = {\"option1\", \"option2\", \"option3\"};\n        String input = args[0];\n        for (String value : allowedValues) {\n            if (value.equals(input)) {\n                System.out.println(\"Valid option: \" + input);\n                return;\n            }\n        }\n        System.out.println(\"Invalid option\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example8 {\n    public static void main(String[] args) {\n        String secret = \"hardcoded-key-12345\";\n        System.out.println(secret);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example9 {\n    public static void main(String[] args) throws Exception {\n        String input = args[0];\n        PreparedStatement pstmt = connection.prepareStatement(\"SELECT * FROM users WHERE id = ?\");\n        pstmt.setString(1, input);\n        pstmt.executeQuery();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example10 {\n    public static void main(String[] args) {\n        String xml = args[0];\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        Document doc = builder.parse(new InputSource(new StringReader(xml)));\n    }\n}",
        "expected": "VULNERABLE"
    }
]
result=evaluate_model("grok",grok_testcases)

In [ ]:
# @title
chatgpt_testcases=[
    {
        "code": "public class Example1 {\n    public static void main(String[] args) {\n        String password = \"admin123\";\n        System.out.println(\"Password is: \" + password);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.sql.*;\npublic class Example2 {\n    public void getUser(String username) throws Exception {\n        Connection conn = DriverManager.getConnection(\"jdbc:mysql://localhost/test\", \"root\", \"\");\n        Statement stmt = conn.createStatement();\n        String query = \"SELECT * FROM users WHERE username = '\" + username + \"'\";\n        ResultSet rs = stmt.executeQuery(query);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example3 {\n    public static void main(String[] args) throws IOException {\n        BufferedReader br = new BufferedReader(new InputStreamReader(System.in));\n        String line = br.readLine();\n        System.out.println(\"You entered: \" + line);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import javax.crypto.Cipher;\nimport javax.crypto.spec.SecretKeySpec;\npublic class Example4 {\n    public void encrypt(String data) throws Exception {\n        byte[] key = \"1234567890123456\".getBytes();\n        SecretKeySpec secretKey = new SecretKeySpec(key, \"AES\");\n        Cipher cipher = Cipher.getInstance(\"AES\");\n        cipher.init(Cipher.ENCRYPT_MODE, secretKey);\n        cipher.doFinal(data.getBytes());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example5 {\n    public static void main(String[] args) {\n        String fileName = args[0];\n        if (!fileName.matches(\"[a-zA-Z0-9]+\\\\\\.txt\")) {\n            throw new IllegalArgumentException(\"Invalid file name\");\n        }\n        System.out.println(\"File name is valid: \" + fileName);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example6 {\n    public void readFile(String filePath) throws IOException {\n        File file = new File(filePath);\n        FileReader fr = new FileReader(file);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example7 {\n    public static void main(String[] args) {\n        int a = 5;\n        int b = 10;\n        int sum = a + b;\n        System.out.println(\"Sum: \" + sum);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.net.*;\npublic class Example8 {\n    public void connect(String host) throws Exception {\n        Socket socket = new Socket(host, 80);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.util.*;\npublic class Example9 {\n    public static void main(String[] args) {\n        List<String> list = new ArrayList<>();\n        list.add(\"A\");\n        list.add(\"B\");\n        System.out.println(list);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.sql.*;\npublic class Example10 {\n    public void safeQuery(String username) throws Exception {\n        Connection conn = DriverManager.getConnection(\"jdbc:mysql://localhost/test\", \"root\", \"\");\n        String query = \"SELECT * FROM users WHERE username = ?\";\n        PreparedStatement ps = conn.prepareStatement(query);\n        ps.setString(1, username);\n        ResultSet rs = ps.executeQuery();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    }
]
result=evaluate_model("chatgpt",chatgpt_testcases)

In [ ]:
# @title
mistral_testcases=[{
        "code": 'public class Example1 {\n    public static void main(String[] args) {\n        String password = "myPassword123";\n        System.out.println("Password is: " + password);\n    }\n}',
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example2 {\n    public static void main(String[] args) throws Exception {\n        String userInput = args[0];\n        String query = "SELECT * FROM users WHERE username = \\\'" + userInput + "\\\'";\n        // Execute query\n    }\n}',
        "expected": "VULNERABLE",
    },
    {
        "code": 'import java.io.File;\n\npublic class Example3 {\n    public static void main(String[] args) throws Exception {\n        String fileName = args[0];\n        File file = new File("data/" + fileName);\n        // Read file\n    }\n}',
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example4 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        System.out.println(userInput);\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'import java.sql.*;\n\npublic class Example5 {\n    public static void main(String[] args) throws Exception {\n        String userInput = args[0];\n        Connection conn = DriverManager.getConnection("jdbc:mysql://localhost/db", "user", "pass");\n        PreparedStatement stmt = conn.prepareStatement("SELECT * FROM users WHERE username = ?");\n        stmt.setString(1, userInput);\n        ResultSet rs = stmt.executeQuery();\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'import java.io.File;\n\npublic class Example6 {\n    public static void main(String[] args) throws Exception {\n        String fileName = args[0];\n        if (!fileName.matches("[a-zA-Z0-9]+\\\\\\\\.txt")) {\n            throw new IllegalArgumentException("Invalid file name");\n        }\n        File file = new File("data/" + fileName);\n        // Read file\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example7 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        if (!userInput.matches("[a-zA-Z0-9]+")) {\n            throw new IllegalArgumentException("Invalid input");\n        }\n        System.out.println(userInput);\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example8 {\n    public static void main(String[] args) {\n        String apiKey = "SECRET_API_KEY_123";\n        System.out.println(apiKey);\n    }\n}',
        "expected": "VULNERABLE",
    },
    {
        "code": 'import java.io.*;\n\npublic class Example9 {\n    public static void main(String[] args) throws IOException {\n        BufferedReader reader = new BufferedReader(new InputStreamReader(System.in));\n        String input = reader.readLine();\n        System.out.println("You entered: " + input);\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example10 {\n    public static void main(String[] args) throws Exception {\n        Runtime.getRuntime().exec(args[0]);\n    }\n}',
        "expected": "VULNERABLE",
    }]

result=evaluate_model("mistral",mistral_testcases)

In [ ]:
all_testcases = []

for item in grok_testcases:
  all_testcases.append(item)
for item in qwen_testcases:
  all_testcases.append(item)
for item in gemini_testcases:
  all_testcases.append(item)
for item in claude_testcases:
  all_testcases.append(item)
for item in chatgpt_testcases:
  all_testcases.append(item)
for item in mistral_testcases:
  all_testcases.append(item)

In [ ]:
len(all_testcases)

In [ ]:
results=evaluate_model("all testcases", all_testcases)

In [ ]:
# @title
grok_new_testcases=[
    {
        "code": "import java.net.URL;\npublic class SSRFVulnerable {\n    public void fetchUrl(String urlString) throws Exception {\n        URL url = new URL(urlString);\n        HttpURLConnection conn = (HttpURLConnection) url.openConnection();\n        conn.connect();\n        BufferedReader reader = new BufferedReader(new InputStreamReader(conn.get19InputStream()));\n        String line;\n        while ((line = reader.readLine()) != null) {\n            System.out.println(line);\n        }\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.net.URL;\npublic class SSRFNotVulnerable {\n    public void fetchUrl(String urlString) throws Exception {\n        if (!urlString.startsWith(\"https://api.example.com/\")) {\n            throw new IllegalArgumentException(\"Only requests to api.example.com are allowed\");\n        }\n        URL url = new URL(urlString);\n        HttpURLConnection conn = (HttpURLConnection) url.openConnection();\n        conn.connect();\n        BufferedReader reader = new BufferedReader(new InputStreamReader(conn.getInputStream()));\n        String line;\n        while ((line = reader.readLine()) != null) {\n            System.out.println(line);\n        }\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class IDORVulnerable {\n    public void viewProfile(HttpServletRequest request, HttpServletResponse response) throws IOException {\n        String userId = request.getParameter(\"id\");\n        String query = \"SELECT * FROM profiles WHERE user_id = '\" + userId + \"'\";\n        Statement stmt = connection.createStatement();\n        ResultSet rs = stmt.executeQuery(query);\n        response.getWriter().write(rs.toString());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class IDORNotVulnerable {\n    public void viewProfile(HttpServletRequest request, HttpServletResponse response) throws IOException {\n        String userId = request.getParameter(\"id\");\n        HttpSession session = request.getSession();\n        String loggedInUser = (String) session.getAttribute(\"user_id\");\n        if (!userId.equals(loggedInUser)) {\n            throw new SecurityException(\"Unauthorized access\");\n        }\n        String query = \"SELECT * FROM profiles WHERE user_id = ?\";\n        PreparedStatement stmt = connection.prepareStatement(query);\n        stmt.setString(1, userId);\n        ResultSet rs = stmt.executeQuery();\n        response.getWriter().write(rs.toString());\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.util.UUID;\npublic class WeakSessionId {\n    public String generateSessionId() {\n        Random rand = new Random(System.currentTimeMillis());\n        return String.valueOf(rand.nextInt(1000000));\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.util.UUID;\npublic class StrongSessionId {\n    public String generateSessionId() {\n        return UUID.randomUUID().toString();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import org.slf4j.Logger;\npublic class SensitiveLogVulnerable {\n    private static final Logger logger = LoggerFactory.getLogger(SensitiveLogVulnerable.class);\n    public void logUserData(String userId, String ssn) {\n        logger.info(\"User {} with SSN {}\", userId, ssn);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import org.slf4j.Logger;\npublic class SensitiveLogNotVulnerable {\n    private static final Logger logger = LoggerFactory.getLogger(SensitiveLogNotVulnerable.class);\n    public void logUserData(String userId) {\n        logger.info(\"User {} accessed system\", userId);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class AccessControlVulnerable {\n    public void deleteUser(HttpServletRequest request) throws Exception {\n        String userId = request.getParameter(\"user_id\");\n        String query = \"DELETE FROM users WHERE id = '\" + userId + \"'\";\n        Statement stmt = connection.createStatement();\n        stmt.executeUpdate(query);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class AccessControlNotVulnerable {\n    public void deleteUser(HttpServletRequest request) throws Exception {\n        HttpSession session = request.getSession();\n        String role = (String) session.getAttribute(\"role\");\n        if (!\"admin\".equals(role)) {\n            throw new SecurityException(\"Only admins can delete users\");\n        }\n        String userId = request.getParameter(\"user_id\");\n        String query = \"DELETE FROM users WHERE id = ?\";\n        PreparedStatement stmt = connection.prepareStatement(query);\n        stmt.setString(1, userId);\n        stmt.executeUpdate();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class ResourceConsumptionVulnerable {\n    public void processData(String input) throws IOException {\n        byte[] buffer = new byte[1024 * 1024 * 100]; // 100MB buffer\n        ByteArrayOutputStream baos = new ByteArrayOutputStream();\n        baos.write(input.getBytes());\n        // Process large data without limits\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class ResourceConsumptionNotVulnerable {\n    public void processData(String input) throws IOException {\n        if (input.length() > 1024 * 10) { // 10KB limit\n            throw new IllegalArgumentException(\"Input too large\");\n        }\n        ByteArrayOutputStream baos = new ByteArrayOutputStream();\n        baos.write(input.getBytes());\n        // Process data with size limit\n    }\n}",
        "expected": "NOT_VULNERABLE"
    }
]
results=evaluate_model("grok new", grok_new_testcases)